
# Notebook 06 — Distribution & Transportation Analysis

## Objective
Evaluate how mathematical optimization changes
transportation decisions, distribution inventory
placement, and regional fulfillment performance.

## Business Questions
1. How did transportation costs change?
2. Which lanes and modes created savings?
3. How did shipment timing change?
4. How did DC inventory levels respond?
5. Were storage capacities and customer demand satisfied?
6. What operational recommendations follow?

## Methodology
Compare the rule-based baseline from Notebook 03
against the validated optimized solution from
Notebook 04.

Analyze transportation lanes, shipping modes,
shipment arrival timing, DC inventory, and cost.

## Outputs
- Lane and mode comparison
- Regional freight cost analysis
- Weekly shipments and receipts
- DC inventory and service analysis
- Executive KPIs and savings decomposition
- Tableau-ready reporting tables

All network parameters are hypothetical.


In [1]:
# STEP 2: Initialize Notebook 06

import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display, Markdown

# Find Project 2 repository
current_path = Path.cwd().resolve()

repo_root = next(
    (
        p for p in [current_path, *current_path.parents]
        if (p / "data/assumptions/bom.csv").is_file()
    ),
    None
)

assert repo_root is not None, "Project 2 repository not found"

# Source datasets
files = {
    "lanes": "data/assumptions/transport_lanes.csv",
    "dcs": "data/assumptions/distribution_centers.csv",
    "inventory": "data/assumptions/initial_inventory.csv",
    "demand": "data/processed/regional_demand_12_weeks.csv",
    "baseline_ship": "outputs/tables/baseline_shipments.csv",
    "optimized_ship": "outputs/tables/optimized_shipments.csv",
    "baseline_dc": "outputs/tables/baseline_dc_inventory.csv",
    "optimized_dc": "outputs/tables/optimized_dc_inventory.csv",
    "baseline_cost": "outputs/tables/baseline_weekly_costs.csv",
    "optimized_cost": "outputs/tables/optimized_cost_breakdown.csv"
}

data = {}

for name, relative_path in files.items():
    path = repo_root / relative_path
    assert path.is_file(), f"Missing file: {path}"
    data[name] = pd.read_csv(path)

# Initial validations
assert len(data) == 10
assert len(data["lanes"]) == 4
assert len(data["dcs"]) == 2
assert len(data["demand"]) == 24
assert data["demand"]["Demand_Units"].sum() == 6180

print("NOTEBOOK 06 INITIALIZED SUCCESSFULLY")
print("=" * 45)
print("Datasets loaded:", len(data))
print("Transportation lanes:", len(data["lanes"]))
print("Distribution centers:", len(data["dcs"]))
print("Total customer demand:",
      data["demand"]["Demand_Units"].sum())

display(pd.DataFrame([
    {"Dataset": name, "Records": len(df)}
    for name, df in data.items()
]))


NOTEBOOK 06 INITIALIZED SUCCESSFULLY
Datasets loaded: 10
Transportation lanes: 4
Distribution centers: 2
Total customer demand: 6180


,Dataset,Records
0,lanes,4
1,dcs,2
2,inventory,6
3,demand,24
4,baseline_ship,22
5,optimized_ship,48
6,baseline_dc,24
7,optimized_dc,24
8,baseline_cost,12
9,optimized_cost,4


In [2]:
# STEP 3: Standardize baseline and optimized shipments

lanes = data["lanes"]

lane_attributes = lanes[[
    "Lane_ID",
    "Destination_DC_ID",
    "Transport_Mode",
    "Cost_Per_Unit_USD",
    "Transit_Time_Weeks",
    "Weekly_Lane_Capacity_Units"
]]

required = [
    "Lane_ID",
    "DC_ID",
    "Ship_Week",
    "Arrival_Week",
    "Ship_Units",
    "Freight_Cost_USD"
]

def prepare_shipments(df, plan):

    assert set(required).issubset(df.columns)

    result = df[required].copy()

    result = result.merge(
        lane_attributes,
        on="Lane_ID",
        how="left",
        validate="many_to_one"
    )

    assert result.notna().all().all()

    # Verify destination and lead-time consistency
    assert (
        result["DC_ID"] ==
        result["Destination_DC_ID"]
    ).all()

    assert (
        result["Ship_Week"] +
        result["Transit_Time_Weeks"] ==
        result["Arrival_Week"]
    ).all()

    # Reconcile original freight costs
    assert np.allclose(
        result["Freight_Cost_USD"],
        result["Ship_Units"] *
        result["Cost_Per_Unit_USD"],
        atol=0.01
    )

    assert result["Ship_Units"].ge(0).all()

    # Check capacity across each lane and order week
    lane_week = result.groupby(
        ["Lane_ID", "Ship_Week"],
        as_index=False
    ).agg(
        Units=("Ship_Units", "sum"),
        Capacity=("Weekly_Lane_Capacity_Units", "first")
    )

    assert (lane_week["Units"] <=
            lane_week["Capacity"]).all()

    # Zero-quantity decisions may extend beyond Week 12.
    # Only active shipments must arrive within the horizon.
    active = result[result["Ship_Units"] > 0]

    assert active["Arrival_Week"].between(1, 12).all()
    assert active["Ship_Week"].between(1, 12).all()

    result["Plan"] = plan

    return result


baseline_shipments = prepare_shipments(
    data["baseline_ship"], "Baseline"
)

optimized_shipments = prepare_shipments(
    data["optimized_ship"], "Optimized"
)

# Combine both plans
all_shipments = pd.concat(
    [baseline_shipments, optimized_shipments],
    ignore_index=True
)

# Active shipments for business analysis
active_shipments = all_shipments.loc[
    all_shipments["Ship_Units"] > 0
].copy()

# Financial reconciliation
baseline_freight = baseline_shipments[
    "Freight_Cost_USD"
].sum()

optimized_freight = optimized_shipments[
    "Freight_Cost_USD"
].sum()

baseline_freight_reference = data["baseline_cost"][
    "Freight_Cost_USD"
].sum()

optimized_freight_reference = data["optimized_cost"].loc[
    data["optimized_cost"]["Cost_Category"] ==
    "Transportation",
    "Cost_USD"
].sum()

assert np.isclose(
    baseline_freight,
    baseline_freight_reference,
    atol=0.01
)

assert np.isclose(
    optimized_freight,
    optimized_freight_reference,
    atol=0.01
)

freight_savings = baseline_freight - optimized_freight

print("TRANSPORTATION DATA VALIDATION")
print("=" * 45)
print("Baseline freight: $", round(baseline_freight, 2))
print("Optimized freight: $", round(optimized_freight, 2))
print("Freight savings: $", round(freight_savings, 2))
print("Lane capacity checks: PASSED")
print("Transit-time checks: PASSED")
print("Freight reconciliation: PASSED")
print("\nSTEP 3: PASSED")


TRANSPORTATION DATA VALIDATION
Baseline freight: $ 34556.5
Optimized freight: $ 25656.5
Freight savings: $ 8900.0
Lane capacity checks: PASSED
Transit-time checks: PASSED
Freight reconciliation: PASSED

STEP 3: PASSED


In [3]:
# STEP 4A: Compare transportation lane decisions

plans = ["Baseline", "Optimized"]

lane_index = pd.MultiIndex.from_product(
    [plans, lanes["Lane_ID"].tolist()],
    names=["Plan", "Lane_ID"]
)

lane_totals = (
    active_shipments.groupby(["Plan", "Lane_ID"])
    .agg(
        Ship_Units=("Ship_Units", "sum"),
        Freight_Cost_USD=("Freight_Cost_USD", "sum")
    )
    .reindex(lane_index, fill_value=0)
    .reset_index()
)

baseline_lanes = lane_totals.loc[
    lane_totals["Plan"] == "Baseline"
].drop(columns="Plan").rename(columns={
    "Ship_Units": "Baseline_Units",
    "Freight_Cost_USD": "Baseline_Cost_USD"
})

optimized_lanes = lane_totals.loc[
    lane_totals["Plan"] == "Optimized"
].drop(columns="Plan").rename(columns={
    "Ship_Units": "Optimized_Units",
    "Freight_Cost_USD": "Optimized_Cost_USD"
})

lane_comparison = baseline_lanes.merge(
    optimized_lanes,
    on="Lane_ID",
    validate="one_to_one"
).merge(
    lanes[[
        "Lane_ID",
        "Destination_DC_ID",
        "Transport_Mode",
        "Cost_Per_Unit_USD",
        "Transit_Time_Weeks"
    ]],
    on="Lane_ID",
    validate="one_to_one"
)

lane_comparison["Unit_Change"] = (
    lane_comparison["Optimized_Units"]
    - lane_comparison["Baseline_Units"]
)

lane_comparison["Freight_Savings_USD"] = (
    lane_comparison["Baseline_Cost_USD"]
    - lane_comparison["Optimized_Cost_USD"]
)

assert len(lane_comparison) == 4

assert np.isclose(
    lane_comparison["Freight_Savings_USD"].sum(),
    freight_savings
)

print("TRANSPORTATION LANE COMPARISON")
display(lane_comparison.round(2))

print("\nTOTAL FREIGHT SAVINGS: $",
      round(freight_savings, 2))
print("STEP 4A: PASSED")


TRANSPORTATION LANE COMPARISON


,Lane_ID,Baseline_Units,Baseline_Cost_USD,Optimized_Units,Optimized_Cost_USD,Destination_DC_ID,Transport_Mode,Cost_Per_Unit_USD,Transit_Time_Weeks,Unit_Change,Freight_Savings_USD
0,LANE_001,3399,11896.5,3019,10566.5,DC_EAST,Standard,3.5,1,-380,1330.0
1,LANE_002,0,0.0,380,2850.0,DC_EAST,Expedited,7.5,0,380,-2850.0
2,LANE_003,0,0.0,2084,10420.0,DC_WEST,Standard,5.0,2,2084,-10420.0
3,LANE_004,2266,22660.0,182,1820.0,DC_WEST,Expedited,10.0,1,-2084,20840.0



TOTAL FREIGHT SAVINGS: $ 8900.0
STEP 4A: PASSED


In [4]:
# STEP 4B: Analyze transportation mode selection

dc_ids = data["dcs"]["DC_ID"].tolist()

mode_index = pd.MultiIndex.from_product(
    [
        plans,
        dc_ids,
        ["Standard", "Expedited"]
    ],
    names=["Plan", "DC_ID", "Transport_Mode"]
)

mode_comparison = (
    active_shipments.groupby(
        ["Plan", "DC_ID", "Transport_Mode"]
    )
    .agg(
        Ship_Units=("Ship_Units", "sum"),
        Freight_Cost_USD=("Freight_Cost_USD", "sum")
    )
    .reindex(mode_index, fill_value=0)
    .reset_index()
)

# Calculate each mode's share of DC shipments
dc_totals = mode_comparison.groupby(
    ["Plan", "DC_ID"]
)["Ship_Units"].transform("sum")

mode_comparison["Mode_Share_Pct"] = np.where(
    dc_totals > 0,
    mode_comparison["Ship_Units"] / dc_totals * 100,
    0
)

print("STANDARD VS EXPEDITED MODE COMPARISON")
display(mode_comparison.round(2))

# Compare freight spending by distribution center
region_totals = (
    active_shipments.groupby(["Plan", "DC_ID"])
    .agg(
        Ship_Units=("Ship_Units", "sum"),
        Freight_Cost_USD=("Freight_Cost_USD", "sum")
    )
    .reset_index()
)

baseline_regions = region_totals.loc[
    region_totals["Plan"] == "Baseline"
][[
    "DC_ID", "Ship_Units", "Freight_Cost_USD"
]].rename(columns={
    "Ship_Units": "Baseline_Units",
    "Freight_Cost_USD": "Baseline_Freight_USD"
})

optimized_regions = region_totals.loc[
    region_totals["Plan"] == "Optimized"
][[
    "DC_ID", "Ship_Units", "Freight_Cost_USD"
]].rename(columns={
    "Ship_Units": "Optimized_Units",
    "Freight_Cost_USD": "Optimized_Freight_USD"
})

region_comparison = baseline_regions.merge(
    optimized_regions,
    on="DC_ID",
    validate="one_to_one"
)

region_comparison["Freight_Savings_USD"] = (
    region_comparison["Baseline_Freight_USD"]
    - region_comparison["Optimized_Freight_USD"]
)

assert np.isclose(
    region_comparison["Freight_Savings_USD"].sum(),
    freight_savings
)

print("\nREGIONAL TRANSPORTATION COMPARISON")
display(region_comparison.round(2))

print("\nSTEP 4B: PASSED")


STANDARD VS EXPEDITED MODE COMPARISON


,Plan,DC_ID,Transport_Mode,Ship_Units,Freight_Cost_USD,Mode_Share_Pct
0,Baseline,DC_EAST,Standard,3399,11896.5,100.00
1,Baseline,DC_EAST,Expedited,0,0.0,0.00
2,Baseline,DC_WEST,Standard,0,0.0,0.00
3,Baseline,DC_WEST,Expedited,2266,22660.0,100.00
4,Optimized,DC_EAST,Standard,3019,10566.5,88.82
5,Optimized,DC_EAST,Expedited,380,2850.0,11.18
6,Optimized,DC_WEST,Standard,2084,10420.0,91.97
7,Optimized,DC_WEST,Expedited,182,1820.0,8.03



REGIONAL TRANSPORTATION COMPARISON


,DC_ID,Baseline_Units,Baseline_Freight_USD,Optimized_Units,Optimized_Freight_USD,Freight_Savings_USD
0,DC_EAST,3399,11896.5,3399,13416.5,-1520.0
1,DC_WEST,2266,22660.0,2266,12240.0,10420.0



STEP 4B: PASSED


In [5]:
# STEP 5A: Build weekly transportation timeline

timeline = pd.MultiIndex.from_product(
    [plans, dc_ids, range(1, 13)],
    names=["Plan", "DC_ID", "Week"]
).to_frame(index=False)

departures = (
    active_shipments.groupby(
        ["Plan", "DC_ID", "Ship_Week"],
        as_index=False
    )
    .agg(
        Departures_Units=("Ship_Units", "sum"),
        Departures_Freight_USD=("Freight_Cost_USD", "sum")
    )
    .rename(columns={"Ship_Week": "Week"})
)

arrivals = (
    active_shipments.groupby(
        ["Plan", "DC_ID", "Arrival_Week"],
        as_index=False
    )["Ship_Units"]
    .sum()
    .rename(columns={
        "Arrival_Week": "Week",
        "Ship_Units": "Arrivals_Units"
    })
)

weekly_transport = timeline.merge(
    departures,
    on=["Plan", "DC_ID", "Week"],
    how="left",
    validate="one_to_one"
).merge(
    arrivals,
    on=["Plan", "DC_ID", "Week"],
    how="left",
    validate="one_to_one"
)

numeric_cols = [
    "Departures_Units",
    "Departures_Freight_USD",
    "Arrivals_Units"
]

weekly_transport[numeric_cols] = (
    weekly_transport[numeric_cols].fillna(0)
)

weekly_transport = weekly_transport.sort_values(
    ["Plan", "DC_ID", "Week"]
).reset_index(drop=True)

# Validate total shipments and receipts
assert len(weekly_transport) == 48

assert np.isclose(
    weekly_transport["Departures_Units"].sum(),
    active_shipments["Ship_Units"].sum()
)

assert np.isclose(
    weekly_transport["Arrivals_Units"].sum(),
    active_shipments["Ship_Units"].sum()
)

print("WEEKLY TRANSPORTATION TIMELINE")
display(weekly_transport.head(15))

print("\nSTEP 5A: PASSED")

WEEKLY TRANSPORTATION TIMELINE


,Plan,DC_ID,Week,Departures_Units,Departures_Freight_USD,Arrivals_Units
0,Baseline,DC_EAST,1,309.0,1081.5,0.0
1,Baseline,DC_EAST,2,309.0,1081.5,309.0
2,Baseline,DC_EAST,3,309.0,1081.5,309.0
3,Baseline,DC_EAST,4,309.0,1081.5,309.0
4,Baseline,DC_EAST,5,309.0,1081.5,309.0
5,Baseline,DC_EAST,6,309.0,1081.5,309.0
6,Baseline,DC_EAST,7,309.0,1081.5,309.0
7,Baseline,DC_EAST,8,309.0,1081.5,309.0
8,Baseline,DC_EAST,9,309.0,1081.5,309.0
9,Baseline,DC_EAST,10,309.0,1081.5,309.0



STEP 5A: PASSED


In [6]:
# STEP 6A: Compare DC inventory and service performance

def prepare_dc_inventory(df, plan):

    result = df.copy()

    # The optimized model enforces full fulfillment,
    # so its exported ledger may omit a shortage field.
    if "Shortage_Units" not in result.columns:
        result["Shortage_Units"] = (
            result["Customer_Demand"]
            - result["Fulfilled_Units"]
        )

    result["Plan"] = plan
    return result


dc_weekly = pd.concat([
    prepare_dc_inventory(
        data["baseline_dc"], "Baseline"
    ),
    prepare_dc_inventory(
        data["optimized_dc"], "Optimized"
    )
], ignore_index=True)

# Add independently calculated transportation arrivals
dc_weekly = dc_weekly.merge(
    weekly_transport[[
        "Plan", "DC_ID", "Week", "Arrivals_Units"
    ]],
    on=["Plan", "DC_ID", "Week"],
    how="left",
    validate="one_to_one"
)

# Add DC capacity and holding-cost parameters
dc_weekly = dc_weekly.merge(
    data["dcs"][[
        "DC_ID",
        "Storage_Capacity_Units",
        "Holding_Cost_Per_Unit_Per_Week_USD"
    ]],
    on="DC_ID",
    how="left",
    validate="many_to_one"
)

assert dc_weekly.notna().all().all()
assert len(dc_weekly) == 48

# Validate shipment arrivals
assert np.allclose(
    dc_weekly["Shipment_Arrivals"],
    dc_weekly["Arrivals_Units"]
)

# Inventory available before customer fulfillment
dc_weekly["Pre_Demand_Inventory"] = (
    dc_weekly["Beginning_Inventory"]
    + dc_weekly["Shipment_Arrivals"]
)

# Storage headroom before demand
dc_weekly["Storage_Headroom_Units"] = (
    dc_weekly["Storage_Capacity_Units"]
    - dc_weekly["Pre_Demand_Inventory"]
)

# Check storage capacity at the correct point in time
assert dc_weekly["Storage_Headroom_Units"].ge(0).all()

# Verify inventory conservation
assert np.allclose(
    dc_weekly["Beginning_Inventory"]
    + dc_weekly["Shipment_Arrivals"]
    - dc_weekly["Fulfilled_Units"],
    dc_weekly["Ending_Inventory"]
)

# Verify customer fulfillment
assert np.allclose(
    dc_weekly["Customer_Demand"],
    dc_weekly["Fulfilled_Units"]
    + dc_weekly["Shortage_Units"]
)

assert dc_weekly["Shortage_Units"].eq(0).all()
assert dc_weekly["Ending_Inventory"].ge(0).all()

# Independently verify regional demand
demand_reference = (
    data["demand"].groupby(
        ["DC_ID", "Week"],
        as_index=False
    )["Demand_Units"].sum()
)

dc_weekly = dc_weekly.merge(
    demand_reference,
    on=["DC_ID", "Week"],
    validate="many_to_one"
)

assert np.allclose(
    dc_weekly["Customer_Demand"],
    dc_weekly["Demand_Units"]
)

# Validate beginning inventory and week-to-week carryover
initial_dc = data["inventory"].loc[
    data["inventory"]["Item_Type"] == "Finished_Good"
].set_index("Location_ID")[
    "Beginning_Inventory_Units"
].to_dict()

for (plan, dc), group in dc_weekly.groupby(
    ["Plan", "DC_ID"]
):
    group = group.sort_values("Week")

    assert np.isclose(
        group.iloc[0]["Beginning_Inventory"],
        initial_dc[dc]
    )

    assert np.allclose(
        group["Beginning_Inventory"].to_numpy()[1:],
        group["Ending_Inventory"].to_numpy()[:-1]
    )

# Calculate inventory holding costs
dc_weekly["Holding_Cost_USD"] = (
    dc_weekly["Ending_Inventory"]
    * dc_weekly["Holding_Cost_Per_Unit_Per_Week_USD"]
)

dc_weekly = dc_weekly.sort_values(
    ["Plan", "DC_ID", "Week"]
).reset_index(drop=True)

print("DISTRIBUTION INVENTORY VALIDATION")
print("=" * 45)
print("Shipment arrival reconciliation: PASSED")
print("Inventory conservation: PASSED")
print("Regional demand reconciliation: PASSED")
print("Storage capacity: PASSED")
print("Customer fulfillment: PASSED")
print("Inventory carryover: PASSED")

display(dc_weekly.head(12))

print("\nSTEP 6A: PASSED")


DISTRIBUTION INVENTORY VALIDATION
Shipment arrival reconciliation: PASSED
Inventory conservation: PASSED
Regional demand reconciliation: PASSED
Storage capacity: PASSED
Customer fulfillment: PASSED
Inventory carryover: PASSED


,Week,DC_ID,Beginning_Inventory,Shipment_Arrivals,Customer_Demand,Fulfilled_Units,Shortage_Units,Ending_Inventory,Plan,Arrivals_Units,Storage_Capacity_Units,Holding_Cost_Per_Unit_Per_Week_USD,Pre_Demand_Inventory,Storage_Headroom_Units,Demand_Units,Holding_Cost_USD
0,1,DC_EAST,330,0,309,309,0,21,Baseline,0.0,450,1.25,330,120,309,26.25
1,2,DC_EAST,21,309,309,309,0,21,Baseline,309.0,450,1.25,330,120,309,26.25
2,3,DC_EAST,21,309,309,309,0,21,Baseline,309.0,450,1.25,330,120,309,26.25
3,4,DC_EAST,21,309,309,309,0,21,Baseline,309.0,450,1.25,330,120,309,26.25
4,5,DC_EAST,21,309,309,309,0,21,Baseline,309.0,450,1.25,330,120,309,26.25
5,6,DC_EAST,21,309,309,309,0,21,Baseline,309.0,450,1.25,330,120,309,26.25
6,7,DC_EAST,21,309,309,309,0,21,Baseline,309.0,450,1.25,330,120,309,26.25
7,8,DC_EAST,21,309,309,309,0,21,Baseline,309.0,450,1.25,330,120,309,26.25
8,9,DC_EAST,21,309,309,309,0,21,Baseline,309.0,450,1.25,330,120,309,26.25
9,10,DC_EAST,21,309,309,309,0,21,Baseline,309.0,450,1.25,330,120,309,26.25



STEP 6A: PASSED


In [7]:
# STEP 6B: Calculate DC service and inventory KPIs

# Average network ending inventory across weeks
network_inventory = (
    dc_weekly.groupby(["Plan", "Week"], as_index=False)
    ["Ending_Inventory"]
    .sum()
)

average_inventory = (
    network_inventory.groupby("Plan")
    ["Ending_Inventory"]
    .mean()
)

kpi_rows = []

for plan in plans:

    records = dc_weekly.loc[
        dc_weekly["Plan"] == plan
    ]

    shipping = active_shipments.loc[
        active_shipments["Plan"] == plan
    ]

    demand_units = records["Customer_Demand"].sum()
    fulfilled_units = records["Fulfilled_Units"].sum()
    expedited_units = shipping.loc[
        shipping["Transport_Mode"] == "Expedited",
        "Ship_Units"
    ].sum()

    kpi_rows.append({
        "Plan": plan,
        "Customer_Demand_Units": demand_units,
        "Fulfilled_Units": fulfilled_units,
        "Service_Level_Pct":
            fulfilled_units / demand_units * 100,
        "Shipped_Units": shipping["Ship_Units"].sum(),
        "Expedited_Units": expedited_units,
        "Expedited_Share_Pct":
            expedited_units / shipping["Ship_Units"].sum() * 100,
        "Avg_Ending_DC_Inventory":
            average_inventory.loc[plan],
        "Min_Storage_Headroom":
            records["Storage_Headroom_Units"].min(),
        "DC_Holding_Cost_USD":
            records["Holding_Cost_USD"].sum(),
        "Freight_Cost_USD":
            shipping["Freight_Cost_USD"].sum()
    })

distribution_kpis = pd.DataFrame(kpi_rows)

assert distribution_kpis["Service_Level_Pct"].eq(100).all()

# Verify holding costs against the financial exports
baseline_holding = data["baseline_cost"][
    "Holding_Cost_USD"
].sum()

optimized_holding = data["optimized_cost"].loc[
    data["optimized_cost"]["Cost_Category"]
    == "DC Inventory Holding",
    "Cost_USD"
].sum()

assert np.isclose(
    distribution_kpis.loc[
        distribution_kpis["Plan"] == "Baseline",
        "DC_Holding_Cost_USD"
    ].iloc[0],
    baseline_holding,
    atol=0.01
)

assert np.isclose(
    distribution_kpis.loc[
        distribution_kpis["Plan"] == "Optimized",
        "DC_Holding_Cost_USD"
    ].iloc[0],
    optimized_holding,
    atol=0.01
)

print("DISTRIBUTION PERFORMANCE KPIs")
display(distribution_kpis.round(2))

print("\nSTEP 6B: PASSED")


DISTRIBUTION PERFORMANCE KPIs


,Plan,Customer_Demand_Units,Fulfilled_Units,Service_Level_Pct,Shipped_Units,Expedited_Units,Expedited_Share_Pct,Avg_Ending_DC_Inventory,Min_Storage_Headroom,DC_Holding_Cost_USD,Freight_Cost_USD
0,Baseline,6180,6180,100.0,5665,2266,40.00,45.0,120,718.2,34556.5
1,Optimized,6180,6180,100.0,5665,562,9.92,7.5,120,119.7,25656.5



STEP 6B: PASSED


In [8]:
# STEP 7A: Reconcile freight savings with network savings

# Cost columns from Notebook 03
baseline_cost_columns = {
    "Procurement": "Procurement_Cost_USD",
    "Manufacturing": "Manufacturing_Cost_USD",
    "Transportation": "Freight_Cost_USD",
    "DC Inventory Holding": "Holding_Cost_USD"
}

# Optimized cost categories from Notebook 04
optimized_category_costs = (
    data["optimized_cost"]
    .set_index("Cost_Category")["Cost_USD"]
    .to_dict()
)

cost_bridge = pd.DataFrame([
    {
        "Cost_Category": category,
        "Baseline_Cost_USD":
            data["baseline_cost"][column].sum(),
        "Optimized_Cost_USD":
            optimized_category_costs[category]
    }
    for category, column in baseline_cost_columns.items()
])

cost_bridge["Savings_USD"] = (
    cost_bridge["Baseline_Cost_USD"]
    - cost_bridge["Optimized_Cost_USD"]
)

baseline_total_cost = cost_bridge[
    "Baseline_Cost_USD"
].sum()

optimized_total_cost = cost_bridge[
    "Optimized_Cost_USD"
].sum()

total_savings = (
    baseline_total_cost - optimized_total_cost
)

savings_pct = (
    total_savings / baseline_total_cost * 100
)

assert np.isclose(
    total_savings,
    cost_bridge["Savings_USD"].sum()
)

assert np.isclose(
    cost_bridge.loc[
        cost_bridge["Cost_Category"] == "Transportation",
        "Savings_USD"
    ].iloc[0],
    freight_savings
)

assert np.isclose(
    baseline_total_cost,
    data["baseline_cost"]["Total_Cost_USD"].sum()
)

assert np.isclose(
    optimized_total_cost,
    data["optimized_cost"]["Cost_USD"].sum()
)

# Tableau-friendly long-format comparison
cost_comparison_long = cost_bridge.melt(
    id_vars="Cost_Category",
    value_vars=[
        "Baseline_Cost_USD",
        "Optimized_Cost_USD"
    ],
    var_name="Plan",
    value_name="Cost_USD"
)

cost_comparison_long["Plan"] = (
    cost_comparison_long["Plan"].replace({
        "Baseline_Cost_USD": "Baseline",
        "Optimized_Cost_USD": "Optimized"
    })
)

print("SUPPLY NETWORK COST SAVINGS BRIDGE")
print("=" * 45)
display(cost_bridge.round(2))

print("\nBaseline cost: $",
      round(baseline_total_cost, 2))
print("Optimized cost: $",
      round(optimized_total_cost, 2))
print("Total savings: $",
      round(total_savings, 2))
print("Cost improvement:",
      round(savings_pct, 2), "%")

print("\nSTEP 7A: PASSED")


SUPPLY NETWORK COST SAVINGS BRIDGE


,Cost_Category,Baseline_Cost_USD,Optimized_Cost_USD,Savings_USD
0,Procurement,254235.0,253725.0,510.0
1,Manufacturing,124630.0,124630.0,0.0
2,Transportation,34556.5,25656.5,8900.0
3,DC Inventory Holding,718.2,119.7,598.5



Baseline cost: $ 414139.7
Optimized cost: $ 404131.2
Total savings: $ 10008.5
Cost improvement: 2.42 %

STEP 7A: PASSED


In [9]:
# STEP 8A: Export transportation reporting datasets

output_dir = repo_root / "outputs" / "tables"
output_dir.mkdir(parents=True, exist_ok=True)

notebook06_outputs = {
    "transport_lane_comparison.csv":
        lane_comparison,

    "transport_mode_comparison.csv":
        mode_comparison,

    "transport_region_comparison.csv":
        region_comparison,

    "transport_weekly_timeline.csv":
        weekly_transport,

    "distribution_inventory_comparison.csv":
        dc_weekly,

    "distribution_performance_kpis.csv":
        distribution_kpis,

    "transport_cost_savings_bridge.csv":
        cost_bridge,

    "transport_cost_comparison_long.csv":
        cost_comparison_long
}

for filename, dataframe in notebook06_outputs.items():

    path = output_dir / filename

    assert len(dataframe) > 0, filename
    dataframe.to_csv(path, index=False)

    assert path.is_file()

    print("Exported:", filename)

print("\nNOTEBOOK 06 EXPORT COMPLETE")
print("Files created:", len(notebook06_outputs))
print("Output directory:", output_dir)

print("\nSTEP 8A: PASSED")


Exported: transport_lane_comparison.csv
Exported: transport_mode_comparison.csv
Exported: transport_region_comparison.csv
Exported: transport_weekly_timeline.csv
Exported: distribution_inventory_comparison.csv
Exported: distribution_performance_kpis.csv
Exported: transport_cost_savings_bridge.csv
Exported: transport_cost_comparison_long.csv

NOTEBOOK 06 EXPORT COMPLETE
Files created: 8
Output directory: /Users/sidkulkarni/Desktop/supply-chain-portfolio/hardware-supply-network-optimizer/outputs/tables

STEP 8A: PASSED


In [10]:
# STEP 8B: Generate Notebook 06 executive summary

from IPython.display import Markdown, display

# Regional and mode insights
leading_region = region_comparison.sort_values(
    "Freight_Savings_USD",
    ascending=False
).iloc[0]

baseline_expedited = mode_comparison.loc[
    (mode_comparison["Plan"] == "Baseline")
    & (mode_comparison["Transport_Mode"] == "Expedited"),
    "Ship_Units"
].sum()

optimized_expedited = mode_comparison.loc[
    (mode_comparison["Plan"] == "Optimized")
    & (mode_comparison["Transport_Mode"] == "Expedited"),
    "Ship_Units"
].sum()

freight_reduction_pct = (
    freight_savings / baseline_freight * 100
)

freight_share_of_savings = (
    freight_savings / total_savings * 100
)

summary = f"""
## Final Summary — Distribution & Transportation Analysis

### Business Objective
Evaluate how optimized transportation planning
changes distribution costs, shipment timing,
inventory availability, and customer service.

### Financial Results
- Baseline network cost: **${baseline_total_cost:,.2f}**
- Optimized network cost: **${optimized_total_cost:,.2f}**
- Total network savings: **${total_savings:,.2f}**
- Network cost improvement: **{savings_pct:.2f}%**

### Transportation Results
- Baseline freight: **${baseline_freight:,.2f}**
- Optimized freight: **${optimized_freight:,.2f}**
- Freight savings: **${freight_savings:,.2f}**
- Freight cost reduction: **{freight_reduction_pct:.2f}%**
- Freight contribution to net savings:
  **{freight_share_of_savings:.2f}%**

### Regional Findings
The region with the largest positive freight
savings contribution was
**{leading_region['DC_ID']}**.

- Contribution:
  **${leading_region['Freight_Savings_USD']:,.2f}**

### Transportation Mode Findings
- Baseline expedited shipments:
  **{baseline_expedited:,.0f} units**
- Optimized expedited shipments:
  **{optimized_expedited:,.0f} units**

### Distribution Performance
- Customer demand: **6,180 units**
- Baseline service level: **100%**
- Optimized service level: **100%**
- Storage capacity checks: **PASSED**
- Shipment arrival reconciliation: **PASSED**
- Inventory conservation: **PASSED**

### Business Interpretation
The optimizer changed the transportation mix
and shipment schedule while preserving full
customer fulfillment and inventory feasibility.

The results demonstrate how transportation,
production timing, and inventory positioning
must be coordinated rather than optimized
independently.

### Limitations
- Hypothetical network costs and capacities
- Historical illustrative demand
- Deterministic weekly planning model
- No real-time shipment tracking
- No uncertainty in transportation lead times
- Initial inventory acquisition costs excluded

### Next Steps
1. Test disruption scenarios.
2. Measure sensitivity to lead times and capacity.
3. Build executive-facing Tableau dashboards.
4. Develop business recommendations.
"""

# Display inside notebook
display(Markdown(summary))

# Save to project documentation
docs_dir = repo_root / "docs"
docs_dir.mkdir(parents=True, exist_ok=True)

summary_file = docs_dir / "notebook06_findings.md"

summary_file.write_text(
    summary,
    encoding="utf-8"
)

assert summary_file.is_file()

print("Executive summary saved:", summary_file)
print("\nNOTEBOOK 06: COMPLETE")



## Final Summary — Distribution & Transportation Analysis

### Business Objective
Evaluate how optimized transportation planning
changes distribution costs, shipment timing,
inventory availability, and customer service.

### Financial Results
- Baseline network cost: **$414,139.70**
- Optimized network cost: **$404,131.20**
- Total network savings: **$10,008.50**
- Network cost improvement: **2.42%**

### Transportation Results
- Baseline freight: **$34,556.50**
- Optimized freight: **$25,656.50**
- Freight savings: **$8,900.00**
- Freight cost reduction: **25.75%**
- Freight contribution to net savings:
  **88.92%**

### Regional Findings
The region with the largest positive freight
savings contribution was
**DC_WEST**.

- Contribution:
  **$10,420.00**

### Transportation Mode Findings
- Baseline expedited shipments:
  **2,266 units**
- Optimized expedited shipments:
  **562 units**

### Distribution Performance
- Customer demand: **6,180 units**
- Baseline service level: **100%**
- Optimized service level: **100%**
- Storage capacity checks: **PASSED**
- Shipment arrival reconciliation: **PASSED**
- Inventory conservation: **PASSED**

### Business Interpretation
The optimizer changed the transportation mix
and shipment schedule while preserving full
customer fulfillment and inventory feasibility.

The results demonstrate how transportation,
production timing, and inventory positioning
must be coordinated rather than optimized
independently.

### Limitations
- Hypothetical network costs and capacities
- Historical illustrative demand
- Deterministic weekly planning model
- No real-time shipment tracking
- No uncertainty in transportation lead times
- Initial inventory acquisition costs excluded

### Next Steps
1. Test disruption scenarios.
2. Measure sensitivity to lead times and capacity.
3. Build executive-facing Tableau dashboards.
4. Develop business recommendations.


Executive summary saved: /Users/sidkulkarni/Desktop/supply-chain-portfolio/hardware-supply-network-optimizer/docs/notebook06_findings.md

NOTEBOOK 06: COMPLETE
